In [2]:
from PyNite import FEModel3D
from PyNite.Visualization import render_model

In [4]:
# def beam_analysis(span,E,G,nu,rho,material,nodes,members,
# beam_model = FEModel3D() 
# E = 200e3     # Modulus of elasticity (MPa)
# G = 77e3      # Shear modulus of elasticity (MPa)
# nu = 0.3      # Poisson's ratio
# rho = 7.7e-6  # Density (kci)
# beam_model.add_material('Steel', E, G, nu, rho)

# beam_model.add_node(name="node1", X=0, Y=0, Z=0) # Change the model by adding nodes
# beam_model.add_node("node2", 12000, 0, 0)
# beam_model.add_node("node3", 15000, 0, 0)


# # beam_model.add_member(name="M1", i_node="node1", j_node="node3", material="Steel", Iy=20e6, Iz=400e6, J=30e3, A=1500)

# # beam_model.def_support("node1", support_DX=True, support_DY=True, support_DZ=True, support_RX=True, support_RY=False, support_RZ=False)
# # beam_model.def_support("node2", False, True, True, False, False, False)

# # beam_model.add_load_combo(name="LC1", factors={"D": 1.4})
# # beam_model.add_load_combo("LC2", {"D": 1.25, "L": 1.5})

# # beam_model.add_member_dist_load(Member="M1", Direction="Fy", w1=-5.5, w2=-5.5, x1=0, x2=15000, case="D")
# # beam_model.add_member_dist_load("M1", "Fy", w1=-7.8, w2=-7.8, x1=0, x2=12000, case="L")
# # beam_model.add_member_dist_load("M1", "Fy", w1=-13.0, w2=-13.0, x1=12000, x2=15000, case="L")

# # beam_model.add_member_pt_load(Member="M1", Direction="Fy", P=-15000, x=4000, case="L")

# # beam_model.analyze(check_statics=True) # Changes the model by performing the analysis and adding analysis results

# # beam_model.Members['M1'].plot_shear(Direction="Fy", combo_name="LC1", n_points=100)
# # beam_model.Members['M1'].plot_moment(Direction="Mz", combo_name="LC1", n_points=100)

# # render_model(beam_model, combo_name='LC2', annotation_size=500)

In [5]:
import csv

def read_beam_file(filename: str) -> list[list[str]]:
    
    """
    Returna a list of list of strings representing the text data in the file at
    'filename'
    """
    csv_acc = [] # File data goes here
    with open(filename, "r") as csv_file:
        csv_reader = csv.reader(csv_file)
        for line in csv_reader:
            csv_acc.append(line)
            
    return csv_acc

In [9]:
sample=read_beam_file("sample beam.txt")

In [10]:
sample

[['Sample Beam'],
 ['4800', '24500', '1200000000', '1', '1'],
 ['1000:P', '3800:R'],
 ['POINT:Fy', '-10000', '4800', 'case:Live'],
 ['DIST:Fy', '30', '30', '0', '4800', 'case:Dead']]

In [7]:
def str_to_float(s:str) -> float|str:
    """
    Converts a string(or a list of strings) to float type.
    if the string passed to the function cannot be converted into a float, then the original string is returned instead.
    """
    try:
        return float(s)
    except ValueError:
        return s

In [8]:
def convert_to_numeric(file_data:list[list[str]])->list[list[float]]:
    """
    Converts all of the numeric data into numbers
    """
    
    numeric_data_final=[]
    
    for data in file_data:
        numeric_data=[]
        for line in data:
            a = str_to_float(line.replace(","," "))
            numeric_data.append(a)
        numeric_data_final.append(numeric_data)

    return numeric_data_final

In [13]:
sample_numeric=convert_to_numeric(sample)

In [14]:
sample_numeric

[['Sample Beam'],
 [4800.0, 24500.0, 1200000000.0, 1.0, 1.0],
 ['1000:P', '3800:R'],
 ['POINT:Fy', -10000.0, 4800.0, 'case:Live'],
 ['DIST:Fy', 30.0, 30.0, 0.0, 4800.0, 'case:Dead']]

In [12]:
def parse_supports(data:list[str])->dict[float,str]:
    """
    Returns a list of suppport details into a dictionary with support locations as keys and support types
    as values where support types can be P(pinned),R(roller) or F(fixed)
    """
    acc={}
    for item in data:
        loc,support = item.split(":")
        acc.update({str_to_float(loc):support})
    return acc

In [16]:
sample_supports=parse_supports(sample_numeric[2])

In [17]:
sample_supports

{1000.0: 'P', 3800.0: 'R'}

In [25]:
def parse_loads(data:list[list[str|float]])->list[dict]:

    """
    Returns the load data in a structured form as list of dicts
    """
    acc=[]
    for item in data:
        type, dirn = item[0].split(":")
        case = item[-1].split(":")[-1]
        if type == "POINT":
            mag=item[1]
            loc=item[2]
            acc.append({"Type": type.title(),
                    "Direction": dirn.title(),
                    "Magnitude": mag,
                    "Location": loc,
                    "Case": case})
            
        elif type == "DIST":
            start_mag=item[1]
            end_mag=item[2]
            start_loc=item[3]
            end_loc=item[4]
            acc.append({"Type": type.title(),
                    "Direction": dirn.title(),
                    "Start Magnitude": start_mag,
                    "End Magnitude": end_mag,
                    "Start Location": start_loc,
                    "End Location": end_loc,
                    "Case": case})
            
    return acc  

In [27]:
parse_loads([sample_numeric[3],sample_numeric[4]])

[{'Type': 'Point',
  'Direction': 'Fy',
  'Magnitude': -10000.0,
  'Location': 4800.0,
  'Case': 'Live'},
 {'Type': 'Dist',
  'Direction': 'Fy',
  'Start Magnitude': 30.0,
  'End Magnitude': 30.0,
  'Start Location': 0.0,
  'End Location': 4800.0,
  'Case': 'Dead'}]

In [28]:
def parse_beam_attributes(data:list[float])->dict[str,float]:
    """
    Returns the list of length and section/material properties of the beam into
    a dictionary which includes L,E,Iz,Iy,A,J,nu,rho
    """
    
    attributes=['L','E','Iz','Iy','A','J','nu','rho']
    acc={}
    for idx,attr in enumerate(attributes):
        try:
            acc.update({attr:data[idx]})
        except IndexError:
            acc.update({attr:1.0})
    return acc
    

In [29]:
parse_beam_attributes(sample_numeric[1])

{'L': 4800.0,
 'E': 24500.0,
 'Iz': 1200000000.0,
 'Iy': 1.0,
 'A': 1.0,
 'J': 1.0,
 'nu': 1.0,
 'rho': 1.0}

In [31]:
def get_structured_beam_data(raw_data: list[list[str]]) -> dict:
    """
    Returns a dictionary that has string keys describing the attributes of a beam for analysis.
    """
    numeric_beam_data = convert_to_numeric(raw_data)
    beam_name = raw_data[0][0]
    beam_attributes = parse_beam_attributes(numeric_beam_data[1])
    supports = numeric_beam_data[2]
    loads = numeric_beam_data[3:]
    structured_data = {}
    structured_data['Name'] = beam_name
    structured_data.update(beam_attributes)
    structured_data['Supports'] = parse_supports(supports)
    structured_data['Loads'] = parse_loads(loads)
    return structured_data

In [35]:
data=get_structured_beam_data(sample)

In [36]:
data

{'Name': 'Sample Beam',
 'L': 4800.0,
 'E': 24500.0,
 'Iz': 1200000000.0,
 'Iy': 1.0,
 'A': 1.0,
 'J': 1.0,
 'nu': 1.0,
 'rho': 1.0,
 'Supports': {1000.0: 'P', 3800.0: 'R'},
 'Loads': [{'Type': 'Point',
   'Direction': 'Fy',
   'Magnitude': -10000.0,
   'Location': 4800.0,
   'Case': 'Live'},
  {'Type': 'Dist',
   'Direction': 'Fy',
   'Start Magnitude': 30.0,
   'End Magnitude': 30.0,
   'Start Location': 0.0,
   'End Location': 4800.0,
   'Case': 'Dead'}]}

In [34]:
def get_node_locations(beam_length:float,supports:list[float])->dict[str,float]:
        
        """
        Returns a dict representing the node number and the node coordinates for the provided 
        support locations and beam length.
        """
        new_nodes = supports[:]
        if 0.0 not in supports:
            new_nodes.append(0.0)
        if beam_length not in supports:
            new_nodes.append(beam_length)
        
        node_locations = {}
        for idx,loc in enumerate(sorted(new_nodes)):
            node_locations.update({f"N{idx}":loc})
        return node_locations 

In [38]:
L = data["L"]
support_loc=list(data['Supports'].keys())
data['Nodes'] = get_node_locations(L,support_loc)
node_dict=data['Nodes']

In [39]:
node_dict

{'N0': 0.0, 'N1': 1000.0, 'N2': 3800.0, 'N3': 4800.0}

In [42]:
def calc_shear_modulus(nu:float, E:float) -> float:
    """
    Returns the shear modulus calculated from 'nu' and 'E'
    """
    G = E / (2 * (1+nu))
    return G

In [43]:
def build_beam (beam_data:dict)->FEModel3D:
    """
    Returns a beam finite element model for the data in 'beam_data' 
    """
    
    beam_model=FEModel3D()
    L = beam_data["L"]
    E = beam_data["E"]
    I = beam_data["Iz"]
    Iy=beam_data["Iy"]
    J = beam_data["J"]
    A = beam_data["A"]
    nu=beam_data["nu"]
    rho=beam_data["rho"]
          
    G=calc_shear_modulus(nu,E)
    beam_model.add_material('default',E,G,nu,rho)

    support_loc=list(beam_data['Supports'].keys())
    beam_data['Nodes'] = get_node_locations(L,support_loc)
    node_dict=beam_data['Nodes']
    for node_no, node_loc in node_dict.items():
        beam_model.add_node(node_no,node_loc,0,0)
        support_type = beam_data['Supports'].get(node_loc, None)
        if support_type == "P":
            beam_model.def_support(node_no, True, True, True, True, False, False)
        elif support_type == "R":
            beam_model.def_support(node_no, False, True, True, False, False, False)
        elif support_type == "F":
            beam_model.def_support(node_no, True, True, True, True, True, True)

    
    beam_model.add_member(beam_data['Name'],"N0",node_no,'default',Iy,I,J,A)
    
    load_cases = []
    for load in beam_data['Loads']:
        if load['Type'] == "Point":
            beam_model.add_member_pt_load(
                beam_data['Name'],
                load['Direction'],
                load['Magnitude'],
                load['Location'],
                case=load["Case"],
            )
            if load['Case'] not in load_cases:
                load_cases.append(load['Case'])
        elif load['Type'] == "Dist":
            beam_model.add_member_dist_load(
                beam_data['Name'],
                load['Direction'],
                load['Start Magnitude'],
                load['End Magnitude'],
                load['Start Location'],
                load['End Location'],
                case=load['Case']
            )
            if load['Case'] not in load_cases:
                load_cases.append(load['Case'])

    for load_case in load_cases:
        beam_model.add_load_combo(load_case, {load_case: 1.0})
    return beam_model
        
        

In [45]:
model=build_beam (data)

In [46]:
model.analyze()